# Clase 115 — Learning rate scheduling

Variar el LR durante el entrenamiento porque ningún LR es óptimo en todas las fases. Estrategias: step, exponential, **cosine annealing** (default moderno) y **warmup + decay** (estándar en Transformers).

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `matplotlib`.

## 1. `ExponentialDecay`: `lr = lr_0 · γ^(step/decay_steps)`

Un schedule se pasa directamente como `learning_rate=` del optimizer.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

exp = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-3, decay_steps=1000, decay_rate=0.9)
print("LR exponencial en steps 0, 1000, 5000:",
      [round(float(exp(s)), 6) for s in (0, 1000, 5000)])
opt = keras.optimizers.Adam(learning_rate=exp)      # el schedule vive dentro del optimizer

## 2. `PiecewiseConstantDecay`: escalones por tramos

Útil para el clásico "cortá el LR a la mitad en las épocas N y M".

In [ ]:
pw = keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[1000, 3000], values=[1e-3, 5e-4, 1e-4])
print("piecewise en steps 500 / 2000 / 4000:",
      [float(pw(s)) for s in (500, 2000, 4000)])

## 3. `CosineDecay` con warmup

Keras 3 soporta warmup nativo: sube linealmente de 0 a `warmup_target` en `warmup_steps` y luego baja con coseno.

In [ ]:
cos = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0, warmup_target=1e-3, warmup_steps=500,
    decay_steps=10_000, alpha=0.0)
lrs = [round(float(cos(s)), 6) for s in (0, 250, 500, 5000, 10_000)]
print("cosine + warmup (sube hasta 1e-3 y luego decae):", lrs)

## 4. `LearningRateScheduler`: función época → LR

Callback que ajusta un LR **escalar** en función de la época (por ejemplo, step decay).

In [ ]:
def step_decay(epoca, lr):
    return lr * 0.5 if (epoca > 0 and epoca % 10 == 0) else lr

lr_callback = keras.callbacks.LearningRateScheduler(step_decay, verbose=0)
print("LearningRateScheduler aplica una función (época, lr) -> nuevo lr")

## 5. `ReduceLROnPlateau`: reactivo

Baja el LR cuando una métrica deja de mejorar (a diferencia del schedule, que es proactivo).

In [ ]:
reduce = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
print("ReduceLROnPlateau baja el LR cuando val_loss se estanca durante 3 épocas")

## 6. Entrenar con cosine + warmup y extraer la curva de LR

In [ ]:
def mlp(learning_rate):
    m = keras.Sequential([
        keras.Input(shape=(784,)),
        layers.Dense(128, activation="relu", kernel_initializer="he_normal"),
        layers.Dense(10,  activation="softmax"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

modelo = mlp(cos)
curva = [float(cos(s)) for s in range(0, 10_000, 500)]
print("modelo con cosine+warmup listo; puntos de la curva LR:", len(curva))
# import matplotlib.pyplot as plt; plt.plot(range(0, 10_000, 500), curva)

## Ejercicios

1. **Schedule básica**: `CosineDecay(...)` dentro de `Adam` y entrenar; graficá `val_loss`.
2. **Visualizar el LR**: evaluá una schedule en varios steps y graficá la curva.
3. **Warmup + Cosine**: compará contra sin warmup en un modelo chico.
4. **ReduceLROnPlateau vs Cosine**: compará el reactivo contra el proactivo.

## Conclusiones

- Un LR fijo arranca bien pero termina demasiado alto para refinar.
- Los schedules (`ExponentialDecay`, `PiecewiseConstantDecay`, `CosineDecay`) se pasan como `learning_rate=` del optimizer.
- **Cosine con warmup** es el estándar moderno (visión y NLP).
- `LearningRateScheduler` (proactivo, por época) y `ReduceLROnPlateau` (reactivo) son alternativas basadas en callbacks; no combinar un schedule del optimizer con estos callbacks.
- Calibrar `decay_steps = epochs · steps_per_epoch` para que no decaiga demasiado rápido.